# Boundary Experiment — Slice B (v2.0)
**P ∈ {2, 8, 16}**, γ ∈ {0.0, 0.3, 0.6, 0.9} (P=2 runs γ={0.6, 0.9} only),
C=21, ρ=0.5, seeds {42, 123, 456, 789, 1011}, modes CI + CD.

CD is infeasible at P ∈ {2, 4} on a T4 (P=2 → 21×511 = 10,731 tokens); those
cells are filtered out, never attempted. At n=5: P=8 and P=16 each 20 CI + 20 CD,
P=2 10 CI only → **90 runs total**.

Output: `results_boundary_P2_P8_P16_acctB_v3.csv` (resume-safe; upload the existing
v3 as a Kaggle dataset input and copy it into /kaggle/working/ before running Cell 7).

Execution order: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8

In [ ]:
# -- Cell 1: Environment ------------------------------------------------------
import gc
import math
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_PATH = Path('/kaggle/working/results_boundary_P2_P8_P16_acctB_v3.csv')

# Schema is fixed and validated on every resume to catch a truncated CSV from a
# session that died mid-write.
SCHEMA = [
    'dataset', 'C', 'rho', 'gamma', 'patch_size', 'mode', 'seed',
    'test_mse', 'test_mae', 'best_epoch', 'batch_size',
    'steps_per_epoch', 'total_steps',
]

print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# -- Cell 1b: Seed resume from existing v3 CSV ---------------------------------
import shutil

# Match by prefix so the truncated suffix or the underscore-vs-hyphen slug
# cannot silently send the run down a fresh-start path.
_RESUME_GLOB = 'results_boundary_P2_P8_P16_acctB_v3*.csv'

if OUT_PATH.exists():
    print(f'{OUT_PATH.name} already in working dir; resume will use it as-is.')
else:
    matches = sorted(Path('/kaggle/input').rglob(_RESUME_GLOB))
    if not matches:
        available = sorted(p.name for p in Path('/kaggle/input').rglob('*.csv'))
        raise FileNotFoundError(
            f'No file matching {_RESUME_GLOB} under /kaggle/input. '
            f'CSV files visible: {available}'
        )
    # If more than one version is mounted, seed from the most complete one.
    counts = {p: len(pd.read_csv(p)) for p in matches}
    src = max(counts, key=counts.get)
    shutil.copy(src, OUT_PATH)
    print(f'Seeded resume from {src.name}: {counts[src]} existing rows.')

In [ ]:
# -- Cell 2: Data generator ---------------------------------------------------
# Leader-follower VAR(1) process.
# Channels 0-9:   leaders   (pure AR(1), phi=0.8)
# Channels 10-19: followers (each follows leader i with coupling gamma)
# Channel 20:     isolate   (pure AR(1), internal negative control)
# Transition matrix A is lower-triangular: spectral radius = phi = 0.8 for all gamma.

N_LEADERS   = 10
N_TOTAL     = 21
PHI         = 0.8
NOISE_STD   = 0.1
N_TIMESTEPS = 20_000


def generate(gamma: float, seed: int) -> np.ndarray:
    """Generate VAR(1) time series of shape (N_TIMESTEPS, N_TOTAL)."""
    rng = np.random.default_rng(seed)
    A = np.zeros((N_TOTAL, N_TOTAL))
    np.fill_diagonal(A, PHI)
    for i in range(N_LEADERS):
        A[i + N_LEADERS, i] = gamma

    X = np.zeros((N_TIMESTEPS, N_TOTAL))
    X[0] = rng.standard_normal(N_TOTAL) * NOISE_STD
    noise = rng.standard_normal((N_TIMESTEPS - 1, N_TOTAL)) * NOISE_STD
    for t in range(1, N_TIMESTEPS):
        X[t] = A @ X[t - 1] + noise[t - 1]
    return X


def split_and_normalise(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Chronological 70/10/20 split with train-statistics normalisation."""
    T       = len(X)
    n_train = int(T * 0.70)
    n_val   = int(T * 0.10)
    train   = X[:n_train]
    val     = X[n_train:n_train + n_val]
    test    = X[n_train + n_val:]
    mean    = train.mean(axis=0, keepdims=True)
    std     = train.std(axis=0, keepdims=True) + 1e-8
    return (train - mean) / std, (val - mean) / std, (test - mean) / std

In [ ]:
# -- Cell 3: Experiment configuration -----------------------------------------
LOOKBACK  = 512
PRED_LEN  = 96
D_MODEL   = 64
N_HEADS   = 8
N_LAYERS  = 3
DROPOUT   = 0.2

BATCH_SIZES   = {'CI': 128, 'CD': 8}
LR            = 1e-4
WARMUP_EPOCHS = 10
MAX_EPOCHS    = 50
PATIENCE      = 10

GAMMAS_FULL    = [0.0, 0.3, 0.6, 0.9]
GAMMAS_PARTIAL = [0.6, 0.9]           # P=2 only: the high-coupling region
SEEDS  = [42, 123, 456, 789, 1011]
MODES  = ['CI', 'CD']

# CD is computationally infeasible at these patch sizes on a T4 (the per-variate
# patch sequence is too long once flattened across variates). Filtered out of the
# grid entirely so a resume can never attempt one and OOM.
CD_INFEASIBLE_PATCH = {2, 4}

# (patch_size, gamma_list)
PATCH_CONFIGS = [
    (8,  GAMMAS_FULL),
    (16, GAMMAS_FULL),
    (2,  GAMMAS_PARTIAL),
]


def is_feasible(mode: str, patch_size: int) -> bool:
    return not (mode == 'CD' and patch_size in CD_INFEASIBLE_PATCH)


def n_patches_for(patch_size: int) -> int:
    return (LOOKBACK - patch_size) // (patch_size // 2) + 1


print('Patch config verification:')
for P, gammas in PATCH_CONFIGS:
    NP    = n_patches_for(P)
    modes = [m for m in MODES if is_feasible(m, P)]
    runs  = len(gammas) * len(SEEDS) * len(modes)
    print(f'  P={P:2d}  S={P // 2}  n_patches={NP:3d}  '
          f'head=Linear({NP * D_MODEL}, {PRED_LEN})  '
          f'modes={modes}  gammas={gammas}  runs={runs}')

total_runs = sum(
    len(g) * len(SEEDS) * len([m for m in MODES if is_feasible(m, P)])
    for P, g in PATCH_CONFIGS
)
print(f'Total runs: {total_runs}')

In [ ]:
# -- Cell 4: Model definitions ------------------------------------------------
def extract_patches(x: torch.Tensor, patch_size: int, stride: int) -> torch.Tensor:
    """x: (B, L) -> (B, N_patches, patch_size)"""
    return x.unfold(dimension=1, size=patch_size, step=stride)


class PatchEmbedding(nn.Module):
    def __init__(self, patch_size: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.proj    = nn.Linear(patch_size, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, N_patches, patch_size) -> (B, N_patches, d_model)"""
        return self.dropout(self.proj(x))


class PatchTST_CI(nn.Module):
    """Channel-independent PatchTST. Each variate processed separately."""

    def __init__(
        self,
        lookback: int,
        pred_len: int,
        n_variates: int,
        patch_size: int,
        stride: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.stride     = stride
        self.n_patches  = (lookback - patch_size) // stride + 1

        self.embed   = PatchEmbedding(patch_size, d_model, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head    = nn.Linear(self.n_patches * d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, L, C) -> (B, pred_len, C)"""
        B, L, C = x.shape
        x_flat  = x.permute(0, 2, 1).reshape(B * C, L)
        patches = extract_patches(x_flat, self.patch_size, self.stride)
        enc     = self.encoder(self.embed(patches))
        out     = self.head(enc.reshape(B * C, -1)).reshape(B, C, -1)
        return out.permute(0, 2, 1)


class PatchTST_CD(nn.Module):
    """Channel-dependent PatchTST. Cross-variate attention over all patches."""

    def __init__(
        self,
        lookback: int,
        pred_len: int,
        n_variates: int,
        patch_size: int,
        stride: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.stride     = stride
        self.n_patches  = (lookback - patch_size) // stride + 1
        self.n_variates = n_variates

        self.embed   = PatchEmbedding(patch_size, d_model, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head    = nn.Linear(self.n_patches * d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, L, C) -> (B, pred_len, C)"""
        B, L, C = x.shape
        x_flat  = x.permute(0, 2, 1).reshape(B * C, L)
        patches = extract_patches(x_flat, self.patch_size, self.stride)
        # (B*C, N_patches, patch_size) -> embed -> (B, C, N_patches, d_model)
        emb = self.embed(patches).reshape(B, C, self.n_patches, -1)
        # flatten C and N_patches for cross-variate attention
        enc = self.encoder(emb.reshape(B, C * self.n_patches, -1))
        # split back per variate, apply head
        flat = enc.reshape(B, C, self.n_patches, -1).reshape(B * C, -1)
        return self.head(flat).reshape(B, C, -1).permute(0, 2, 1)


def build_model(mode: str, patch_size: int) -> nn.Module:
    stride = patch_size // 2
    cls    = PatchTST_CI if mode == 'CI' else PatchTST_CD
    return cls(
        LOOKBACK, PRED_LEN, N_TOTAL, patch_size, stride,
        D_MODEL, N_HEADS, N_LAYERS, DROPOUT,
    ).to(DEVICE)


# -- Verification --------------------------------------------------------------
print('Head verification:')
for P, _ in PATCH_CONFIGS:
    NP = n_patches_for(P)
    for mode in MODES:
        if not is_feasible(mode, P):
            continue
        head = build_model(mode, P).head
        assert head.in_features == NP * D_MODEL, f'{mode} head mismatch at P={P}'
        assert head.out_features == PRED_LEN,    f'{mode} head out mismatch at P={P}'
    print(f'  P={P:2d}: Linear({NP * D_MODEL}, {PRED_LEN})  OK')

In [ ]:
# -- Cell 5: Dataset construction ---------------------------------------------
# Lazy sliding-window dataset. Windows overlap by >99%, so materialising every
# window densely (the previous approach) wasted ~0.7 GB per run for the train
# split alone. Here only the base (T, C) array is held; each window is a view
# sliced on access. Numerically identical to dense stacking; default_collate
# stacks the per-item slices into the same (B, L, C) / (B, H, C) batches.
class WindowDataset(Dataset):
    def __init__(self, data: np.ndarray, lookback: int, pred_len: int) -> None:
        self.data     = torch.from_numpy(np.ascontiguousarray(data, dtype=np.float32))
        self.lookback = lookback
        self.pred_len = pred_len
        self.n        = len(data) - lookback - pred_len + 1
        if self.n <= 0:
            raise ValueError(
                f'Not enough timesteps ({len(data)}) for lookback={lookback} '
                f'+ pred_len={pred_len}.'
            )

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor]:
        x = self.data[i : i + self.lookback]
        y = self.data[i + self.lookback : i + self.lookback + self.pred_len]
        return x, y

In [ ]:
# -- Cell 6: Training functions -----------------------------------------------
def cosine_lr_with_warmup(
    optimizer: torch.optim.Optimizer,
    epoch: int,
    warmup_epochs: int,
    max_epochs: int,
    base_lr: float,
) -> None:
    """In-place LR update. Linear warmup then cosine decay to 1e-6."""
    min_lr = 1e-6
    if epoch < warmup_epochs:
        lr = base_lr * (epoch + 1) / warmup_epochs
    else:
        progress = (epoch - warmup_epochs) / max(1, max_epochs - warmup_epochs)
        lr = min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))
    for pg in optimizer.param_groups:
        pg['lr'] = lr


def _ckpt_path(mode: str, gamma: float, seed: int, patch_size: int) -> str:
    """Per-run checkpoint path, in the same directory as the output CSV. Unique
    per run so a NaN/failed run can never load another run's weights."""
    return str(OUT_PATH.parent / f'_ckpt_P{patch_size}_g{gamma}_{mode}_s{seed}.pt')


def train_one_run(mode: str, gamma: float, seed: int, patch_size: int) -> dict:
    """Full training run. Returns result dict. Frees its own GPU memory before
    returning so 90 sequential runs do not fragment VRAM."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed(seed)

    ckpt = _ckpt_path(mode, gamma, seed, patch_size)

    X_raw = generate(gamma, seed)
    train_data, val_data, test_data = split_and_normalise(X_raw)

    batch_size   = BATCH_SIZES[mode]
    train_loader = DataLoader(
        WindowDataset(train_data, LOOKBACK, PRED_LEN),
        batch_size=batch_size, shuffle=True, drop_last=True,
    )
    val_loader = DataLoader(
        WindowDataset(val_data, LOOKBACK, PRED_LEN),
        batch_size=batch_size * 4, shuffle=False,
    )

    model     = build_model(mode, patch_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scaler    = GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))
    criterion = nn.MSELoss()

    best_val_loss    = float('inf')
    best_epoch       = 0
    best_total_steps = 0
    patience_counter = 0
    steps_so_far     = 0
    steps_per_epoch  = len(train_loader)
    ckpt_written     = False

    for epoch in range(MAX_EPOCHS):
        cosine_lr_with_warmup(optimizer, epoch, WARMUP_EPOCHS, MAX_EPOCHS, LR)

        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            with autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            steps_so_far += 1

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                with autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                    val_loss += criterion(model(xb), yb).item() * len(xb)
        val_loss /= len(val_loader.dataset)

        # NaN/inf guard: an unstable epoch must not silently masquerade as 'not
        # better' and leave the run with no checkpoint.
        if not math.isfinite(val_loss):
            print(f'    [warn] non-finite val_loss at epoch {epoch + 1}; '
                  f'stopping this run early.', flush=True)
            break

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_epoch       = epoch + 1
            best_total_steps = steps_so_far
            patience_counter = 0
            torch.save(model.state_dict(), ckpt)
            ckpt_written = True
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    if not ckpt_written:
        raise RuntimeError(
            f'No checkpoint written for P={patch_size} gamma={gamma} '
            f'mode={mode} seed={seed} (all epochs non-finite or no improvement). '
            f'Run not recorded.'
        )

    model.load_state_dict(torch.load(ckpt, weights_only=True))
    model.eval()
    test_loader = DataLoader(
        WindowDataset(test_data, LOOKBACK, PRED_LEN),
        batch_size=batch_size * 4, shuffle=False,
    )
    test_mse = test_mae = 0.0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            with autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                pred = model(xb)
            test_mse += nn.functional.mse_loss(pred.float(), yb, reduction='sum').item()
            test_mae += nn.functional.l1_loss(pred.float(),  yb, reduction='sum').item()
    n_test    = len(test_loader.dataset) * PRED_LEN * N_TOTAL
    test_mse /= n_test
    test_mae /= n_test

    result = {
        'dataset':         'leader_follower_var1',
        'C':               N_TOTAL,
        'rho':             0.5,
        'gamma':           gamma,
        'patch_size':      patch_size,
        'mode':            mode,
        'seed':            seed,
        'test_mse':        round(test_mse, 8),
        'test_mae':        round(test_mae, 8),
        'best_epoch':      best_epoch,
        'batch_size':      batch_size,
        'steps_per_epoch': steps_per_epoch,
        'total_steps':     best_total_steps,
    }

    # Free this run's memory before the next one.
    del model, optimizer, scaler, train_loader, val_loader, test_loader
    if os.path.exists(ckpt):
        os.remove(ckpt)
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    return result

In [ ]:
# -- Cell 7: Main training loop -----------------------------------------------
# Order: P=8 and P=16 first (cheaper), P=2 last (most expensive).
# Resume skips any (patch_size, gamma, mode, seed) already in the CSV.
# CD-infeasible cells are filtered out and never attempted.
EXPERIMENTS = [
    (patch_size, gamma, mode, seed)
    for patch_size, gammas in PATCH_CONFIGS
    for gamma in gammas
    for mode  in MODES
    for seed  in SEEDS
    if is_feasible(mode, patch_size)
]


def load_completed(path: Path) -> set:
    """Read finished runs from the CSV. Validates schema and drops a trailing
    truncated row from a session that died mid-append, rewriting the clean file."""
    if not (path.exists() and path.stat().st_size > 100):
        print('Starting fresh.')
        return set()
    try:
        existing = pd.read_csv(path, on_bad_lines='skip')
    except pd.errors.ParserError as exc:
        raise ValueError(f'Cannot parse {path.name}: {exc}') from exc
    missing = set(SCHEMA) - set(existing.columns)
    if missing:
        raise ValueError(f'CSV missing columns {missing}; refusing to resume.')
    # A row truncated mid-write parses as a short line that pandas pads with NaN
    # in the trailing columns. Drop any row with NaN in a key field; these are
    # partial writes, not completed runs, and must be re-run rather than skipped.
    n_before = len(existing)
    existing = existing.dropna(subset=SCHEMA).reset_index(drop=True)
    if len(existing) < n_before:
        existing.to_csv(path, index=False)
        print(f'[recover] dropped {n_before - len(existing)} malformed row(s) '
              f'from {path.name} and rewrote it clean.')
    done = {
        (int(r['patch_size']), float(r['gamma']), str(r['mode']), int(r['seed']))
        for _, r in existing.iterrows()
    }
    print(f'Resuming: {len(done)} runs already complete.')
    return done


def append_row(path: Path, result: dict) -> None:
    """Atomic append: write the full table to a temp file then os.replace, so a
    crash mid-write can never corrupt the canonical CSV."""
    row_df = pd.DataFrame([result], columns=SCHEMA)
    if path.exists() and path.stat().st_size > 100:
        combined = pd.concat([pd.read_csv(path), row_df], ignore_index=True)
    else:
        combined = row_df
    tmp = path.with_suffix('.tmp')
    combined.to_csv(tmp, index=False)
    os.replace(tmp, path)


completed = load_completed(OUT_PATH)
remaining = [e for e in EXPERIMENTS if e not in completed]
total     = len(EXPERIMENTS)
done      = len(completed)
print(f'{len(remaining)} runs remaining of {total}.')

for patch_size, gamma, mode, seed in remaining:
    done += 1
    label = f'[{done}/{total}] P={patch_size} gamma={gamma} mode={mode} seed={seed}'
    print(f'{label} ...', flush=True)
    t0 = time.time()

    result  = train_one_run(mode, gamma, seed, patch_size)
    elapsed = int(time.time() - t0)

    print(
        f'{label} test_mse={result["test_mse"]:.4f}  '
        f'best_epoch={result["best_epoch"]}  ({elapsed}s)',
        flush=True,
    )
    append_row(OUT_PATH, result)
    gc.collect()
    torch.cuda.empty_cache()

print(f'\nDone. Results saved to {OUT_PATH}')
print(f'Total runs: {total}')

In [ ]:
# -- Cell 8: Sanity check and summary -----------------------------------------
df = pd.read_csv(OUT_PATH)

print(f'Rows: {len(df)}  (expected {total})')
print(f'Patch sizes present: {sorted(df["patch_size"].unique())}')
print(f'Gammas present:      {sorted(df["gamma"].unique())}')
print(f'Seeds present:       {sorted(df["seed"].unique())}')

print('\n=== Mean test MSE per (patch_size, gamma, mode) ===')
summary = (
    df.groupby(['patch_size', 'gamma', 'mode'])['test_mse']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
print(summary.to_string(index=False))

print('\n=== CD/CI ratio per (patch_size, gamma) ===')
for P in sorted(df['patch_size'].unique()):
    sub     = summary[summary['patch_size'] == P]
    ci_mean = sub[sub['mode'] == 'CI'].set_index('gamma')['mean']
    cd_mean = sub[sub['mode'] == 'CD'].set_index('gamma')['mean']
    for g in sorted(ci_mean.index):
        if g in cd_mean.index:
            ratio = cd_mean[g] / ci_mean[g]
            print(f'  P={P:2d}  gamma={g}: CI={ci_mean[g]:.4f}  '
                  f'CD={cd_mean[g]:.4f}  ratio={ratio:.4f}')
        else:
            print(f'  P={P:2d}  gamma={g}: CI={ci_mean[g]:.4f}  CD=N/A (infeasible)')